In [1]:
from libraries.inference_training import Configuration, ImageDataset
from libraries.inference_training import initCudaEnvironment, createTransforms
from libraries.inference_training import drawImageAndFeatureMasks
from libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from libraries.inference_training import trainModel, saveModel, loadModel
from libraries.inference_training import createModelInstance, testInference
from libraries.engine import evaluate
import libraries.utils as utils
import torch
import os
import random

In [2]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# load model

In [3]:
def load_model(path:str, config=None):
    """
    :param path: path naar model save location 
    :param config: config als het eerder is ingesteld
    :return: 
    """
    if not config:
        config = Configuration()
    
    model = createModelInstance(config)
    loadModel(config, model, path)
    
    return model, config

In [4]:
def load_default_config():
    config = Configuration()
    config.setIsCrowd(False)
    config.setFilePrefix("")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.addLegendEntry("Background", 0, "#00000000")
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    
    return config

# average recall/average precision evaluation based on different IOU prerequisites template

In [ ]:
model_path = "<insert path to model here>"
eval_path = "<insert path to evaluation dataset here>"
config = load_default_config()
config.setDatasetPaths(testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)

TODO: DICE score evaluation template, mAP score evaluation template

# 25 epoch combo model evaluation

In [11]:
model_path = "C:/xxx/models/combo_models/combo_sets_model.pt"
trainPath = "C:/xxx/datasets/combo_overlay_sets/train"
eval_path = "C:/xxx/datasets/combo_overlay_sets/eval"
config = load_default_config()
config.setDatasetPaths(trainPath= trainPath, testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)

creating index...
index created!
Test:  [  0/400]  eta: 0:05:21  model_time: 0.7809 (0.7809)  evaluator_time: 0.0050 (0.0050)  time: 0.8029  data: 0.0149  max mem: 773
Test:  [100/400]  eta: 0:02:41  model_time: 0.5196 (0.5195)  evaluator_time: 0.0060 (0.0063)  time: 0.5374  data: 0.0124  max mem: 773
Test:  [200/400]  eta: 0:01:47  model_time: 0.5128 (0.5174)  evaluator_time: 0.0050 (0.0061)  time: 0.5369  data: 0.0138  max mem: 773
Test:  [300/400]  eta: 0:00:53  model_time: 0.5116 (0.5168)  evaluator_time: 0.0040 (0.0059)  time: 0.5297  data: 0.0118  max mem: 773
Test:  [399/400]  eta: 0:00:00  model_time: 0.5206 (0.5171)  evaluator_time: 0.0070 (0.0060)  time: 0.5388  data: 0.0125  max mem: 773
Test: Total time: 0:03:34 (0.5365 s / it)
Averaged stats: model_time: 0.5206 (0.5171)  evaluator_time: 0.0070 (0.0060)
Accumulating evaluation results...
DONE (t=0.04s).
Accumulating evaluation results...
DONE (t=0.04s).
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   al

# 25 epoch augmented combo model evaluation

In [10]:
model_path = "C:/xxx/models/combo_models/augment/combo_model_with_augment_epoch_25.pt"
trainPath = "C:/xxx/datasets/combo_overlay_sets/train"
eval_path = "C:/xxx/datasets/combo_overlay_sets/eval"
config = load_default_config()
config.setDatasetPaths(trainPath=trainPath, testPath=eval_path)
evalDataset = ImageDataset(config, False, createTransforms(False))
config = load_default_config()
model, config = load_model(model_path,config)
testDataLoader = torch.utils.data.DataLoader(
    evalDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config.device)

creating index...
index created!
Test:  [  0/400]  eta: 0:04:07  model_time: 0.6033 (0.6033)  evaluator_time: 0.0030 (0.0030)  time: 0.6183  data: 0.0080  max mem: 688
Test:  [100/400]  eta: 0:02:34  model_time: 0.5047 (0.4948)  evaluator_time: 0.0040 (0.0060)  time: 0.5329  data: 0.0164  max mem: 688
Test:  [200/400]  eta: 0:01:44  model_time: 0.5066 (0.5023)  evaluator_time: 0.0040 (0.0064)  time: 0.5292  data: 0.0157  max mem: 688
Test:  [300/400]  eta: 0:00:52  model_time: 0.5047 (0.5051)  evaluator_time: 0.0040 (0.0066)  time: 0.5340  data: 0.0168  max mem: 688
Test:  [399/400]  eta: 0:00:00  model_time: 0.5103 (0.5068)  evaluator_time: 0.0050 (0.0067)  time: 0.5358  data: 0.0182  max mem: 688
Test: Total time: 0:03:32 (0.5305 s / it)
Averaged stats: model_time: 0.5103 (0.5068)  evaluator_time: 0.0050 (0.0067)
Accumulating evaluation results...
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.05s).
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   al